In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score

In [2]:
train = pd.read_csv(Path('home-data-for-ml-course/train.csv'))
test = pd.read_csv(Path('home-data-for-ml-course/test.csv'))

In [3]:
y=np.log1p(train['SalePrice'])
n_train = len(train)

In [4]:
all_data = pd.concat([train.drop(columns=['SalePrice']), test], axis=0, ignore_index=True)

In [5]:

none_cols = ['PoolQC','MiscFeature','Alley','Fence','FireplaceQu',
             'GarageType','GarageFinish','GarageQual','GarageCond',
             'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2']
for col in none_cols:
    all_data[col] = all_data[col].fillna('None')

In [6]:
zero_cols = ['GarageYrBlt', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2',
             'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath',
             'GarageCars', 'GarageArea']
for col in zero_cols:
    all_data[col] = all_data[col].fillna(0)

In [7]:
all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'] \
    .transform(lambda x: x.fillna(x.median()))

In [8]:
mode_cols = ['MSZoning', 'Functional', 'Electrical', 'Utilities',
             'Exterior1st', 'Exterior2nd', 'KitchenQual', 'SaleType']
for col in mode_cols:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

In [9]:
# Report remaining missing values and fill them sensibly
missing = all_data.isnull().sum()
missing = missing[missing > 0]
print("Missing columns before fill:\n", missing)
for col in missing.index:
    if pd.api.types.is_numeric_dtype(all_data[col]):
        all_data[col] = all_data[col].fillna(all_data[col].median())
    else:
        all_data[col] = all_data[col].fillna(all_data[col].mode()[0])
# verify all filled
assert all_data.isnull().sum().sum() == 0, "Still missing values after fill!"

Missing columns before fill:
 MasVnrType    1766
dtype: int64


In [10]:
all_data_encoded = pd.get_dummies(all_data)
X = all_data_encoded.iloc[:n_train, :]
X_test = all_data_encoded.iloc[n_train:, :]

In [ ]:
model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = -cross_val_score(model, X, y, cv=kf, scoring='neg_root_mean_squared_error')
print(f"CV RMSE (log scale): {scores.mean():.4f} (+/- {scores.std():.4f})")

In [ ]:
model.fit(X, y)
log_preds = model.predict(X_test)
preds = np.expm1(log_preds)

In [ ]:
submission = pd.DataFrame({'Id': test['Id'], 'SalePrice': preds})
submission.to_csv('submission.csv', index=False)
submission.head()